# Daily Challenge – Week 6, Day 1: Classification with Neural Networks in TensorFlow
Run in **Google Colab** (TensorFlow pre-installed).

## 1. Classification Types
- **Binary classification:** each example belongs to **one of two** classes. *Example:* is an email spam or not spam? Output: one probability (sigmoid).
- **Multi-class classification:** each example belongs to **exactly one of three or more** classes. *Example:* which digit (0–9) is in an image? Output: one probability per class, adding up to 1 (softmax).
- **Multi-label classification:** each example can have **several labels at the same time**. *Example:* tagging a photo as "beach", "sunset" **and** "people". Output: an independent probability per label (sigmoid on each).

## 2. Set Up the Environment and Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split

tf.random.set_seed(42)

samples = 1000
X, y = make_circles(samples, noise=0.03, random_state=42)
print("X:", X[:5])
print("y:", y[:5])

circles = pd.DataFrame({"X0": X[:, 0], "X1": X[:, 1], "label": y})
print(circles["label"].value_counts())

plt.figure(figsize=(6, 6))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.RdYlBu, s=12)
plt.title("make_circles: two classes, one circle inside the other")
plt.show()

The two classes form an **inner and an outer circle**, so **no straight line can separate them**. A model needs non-linearity to solve this.

## 3. Build a Basic Neural Network Model (one dense layer)

In [ ]:
model_1 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(2,)),
    tf.keras.layers.Dense(1),
])
model_1.compile(loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
                optimizer=tf.keras.optimizers.SGD(),
                metrics=["accuracy"])
model_1.fit(X, y, epochs=5, verbose=0)
print("Model 1 – loss, accuracy:", model_1.evaluate(X, y, verbose=0))

Accuracy is about **50%**, no better than guessing. A single linear layer can only draw a straight line.

## 4. Improve the Model: more layers, more epochs, Adam

In [ ]:
model_2 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(2,)),
    tf.keras.layers.Dense(100),
    tf.keras.layers.Dense(10),
    tf.keras.layers.Dense(1),
])
model_2.compile(loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])
model_2.fit(X, y, epochs=100, verbose=0)
print("Model 2 – loss, accuracy:", model_2.evaluate(X, y, verbose=0))

## 5. Visualize the Decision Boundary

In [ ]:
def plot_decision_boundary(model, X, y, title=""):
    """Colour the plane by the model's prediction and overlay the real points."""
    x_min, x_max = X[:, 0].min() - 0.1, X[:, 0].max() + 0.1
    y_min, y_max = X[:, 1].min() - 0.1, X[:, 1].max() + 0.1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
    grid = np.c_[xx.ravel(), yy.ravel()]
    pred = model.predict(grid, verbose=0)
    pred = (pred > 0.5).astype(int).reshape(xx.shape)
    plt.contourf(xx, yy, pred, cmap=plt.cm.RdYlBu, alpha=0.7)
    plt.scatter(X[:, 0], X[:, 1], c=y, s=12, cmap=plt.cm.RdYlBu, edgecolors="k", linewidths=0.2)
    plt.title(title)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plot_decision_boundary(model_1, X, y, "Model 1: one dense layer")
plt.subplot(1, 2, 2)
plot_decision_boundary(model_2, X, y, "Model 2: 3 layers, no activations")
plt.show()

Both boundaries are **straight lines**. Adding layers didn't help, because without activation functions, stacking linear layers still gives a linear model. (Note: these models output logits, so "> 0.5" isn't the exact boundary, but the line shape is the point here.) We need **non-linear activation functions**.

## 6. Incorporate Activation Functions (ReLU and Sigmoid)

In [ ]:
model_3 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(2,)),
    tf.keras.layers.Dense(4, activation="relu"),
    tf.keras.layers.Dense(4, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),   # outputs a probability between 0 and 1
])
model_3.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
                metrics=["accuracy"])
history_3 = model_3.fit(X, y, epochs=100, verbose=0)
print("Model 3 – loss, accuracy:", model_3.evaluate(X, y, verbose=0))

plt.figure(figsize=(6, 5))
plot_decision_boundary(model_3, X, y, "Model 3: ReLU hidden layers + sigmoid output")
plt.show()

With **ReLU** in the hidden layers and **sigmoid** at the output, the network can bend its decision boundary into a closed shape around the inner circle, and accuracy jumps close to **100%**.

## 7. Split Data into Training and Testing Sets (80 / 20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train:", X_train.shape, "| Test:", X_test.shape)

model_4 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(2,)),
    tf.keras.layers.Dense(4, activation="relu"),
    tf.keras.layers.Dense(4, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
model_4.compile(loss="binary_crossentropy",
                optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
                metrics=["accuracy"])
history_4 = model_4.fit(X_train, y_train, epochs=25, validation_split=0.1, verbose=0)

## 8. Evaluate and Visualize Final Model Performance

In [ ]:
test_loss, test_acc = model_4.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.2%}")

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plot_decision_boundary(model_4, X_train, y_train, "Training data")
plt.subplot(1, 2, 2)
plot_decision_boundary(model_4, X_test, y_test, "Test data")
plt.show()

pd.DataFrame(history_4.history).plot(figsize=(8, 4), title="Model 4 – loss and accuracy per epoch")
plt.xlabel("Epoch")
plt.show()

## 9. Key Takeaways
- **Visualise the data first.** The scatter plot immediately showed that a straight line couldn't separate the classes, which explained why the first models were stuck at ~50%.
- **More layers alone aren't enough.** Without activation functions, a deep network is still linear. **Non-linear activations (ReLU)** are what let neural networks learn curved, complex boundaries.
- **The output activation should fit the task:** sigmoid for binary classification (softmax for multi-class).
- **Hyperparameters matter:** switching from SGD to **Adam**, increasing the learning rate to 0.01, and training for more epochs all sped up learning a lot.
- **Always evaluate on unseen test data** and plot the decision boundary for both training and test sets. Similar performance on both means the model generalises rather than memorises.